In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('fpb_test.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,text,label,source
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank


### libraries

'transformers' - for model
'accelerate' - optimise model loading

In [ ]:
# install transformers
!pip install transformers accelerate -q

### huggingface login

run this in colab with HF_TOKEN as a secret

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the Hugging Face token from Colab's secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
else:
    print("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")

Successfully logged in to Hugging Face.


### load llama model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# model id
model_name = "meta-llama/llama-3.2-3B"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

### data prep, few-shot

create a prompt that includes a few examples of the task (the 'shots') and then the text we want the model to classify

split the dataset into a small set for examples and the rest for classification

In [ ]:
# unique labels are...
print("Unique labels in the dataset:", df['label'].unique())
unique_labels = df['label'].unique().tolist()

# Shuffle the dataframe to ensure random selection of few-shot examples
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Select a few examples for few-shot prompting
num_few_shot_examples = 5 # 10 would be too much and take too long
few_shot_examples = df_shuffled.head(num_few_shot_examples)
inference_data = df_shuffled.tail(len(df_shuffled) - num_few_shot_examples).reset_index(drop=True)

# Define the prompt prefix using the few-shot examples // OUTDATED, SEE BELOW

# few_shot_prompt_prefix = f"Classify the following text into one of these categories: {', '.join(unique_labels)}.\nThe first few samples have been pre-filled for you.\n\n"

# for index, row in few_shot_examples.iterrows():
#     few_shot_prompt_prefix += f"Text: '{row['text']}'\nLabel: {row['label']}\n\n"

# print(f"\nPrompt: {few_shot_prompt_prefix}")
# print(f"Number of inference samples: {len(inference_data)}")

Unique labels in the dataset: ['Neutral' 'Fear' 'Optimism' 'Sadness' 'Joy']


### few-shot inference

iterate through inference_data

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# classify function
def classify(text, candidate_labels):

    few_shot_prompt_prefix = f"Your task is to classify text into one of the following categories: {', '.join(unique_labels)}. The first few samples have been pre-filled for you:\n\n"

    for index, row in few_shot_examples.iterrows():
        few_shot_prompt_prefix += f"Example Text: '{row['text']}'\nExample Label: {row['label']}\n\n"

    # construct in chat format (like gemma)
    chat_prompt = [
        {"role": "user", "content": few_shot_prompt_prefix + f"Now, based on these examples, please categorise this text.\n\nText: {text}\nCategory:"}
    ]

    prompt = tokenizer.apply_chat_template(chat_prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # high max new tokens value helps us guarantee outputs for longer prompts such as this one
    output = model.generate(**inputs, max_new_tokens=50, do_sample=False)

    # decode only the newly generated tokens, skipping the input prompt
    generated_token_ids = output[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()

    print(f"  Generated text for \'{text[:50]}...\': '{generated_text}'") # debug

    predicted_label = "Unknown" # default value

    # remove any "category" labels that may be left in the generated text
    cleaned_generated_text = generated_text.replace("Category:", "").strip()

    # case-insensitive matching
    generated_text_lower = cleaned_generated_text.lower()

    # Attempt to find the best match by checking if the label is *in* the generated text
    for label in candidate_labels:
        if label.lower() in generated_text_lower:
            predicted_label = label
            break

    # If still no match after all attempts, and cleaned_generated_text is not empty, use it as a fallback
    if predicted_label == "Unknown" and cleaned_generated_text:
        predicted_label = cleaned_generated_text

    return predicted_label

# time to evaluate!
test_inference = inference_data.head(10)

y_true = inference_data['label'].tolist() # all the correct labels
y_pred = [classify(text, unique_labels) for text in inference_data['text']] # all the predicted labels

# print(y_true)
# print(y_pred)

# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

# export metrics
results_dict = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [accuracy, precision, recall, f1]
}

# create dataframe
results_df = pd.DataFrame(results_dict)

# export to csv
results_df.to_csv('few_shot_llama_01.csv', index=False)

print("Results saved to 'few_shot_llama_01.csv'")
display(results_df)